In [ ]:
!python -m pip install --upgrade pip
!pip install -U bitsandbytes transformers peft datasets hf_transfer trl wandb
!pip install flash-attn --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 85.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 103.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 77.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 97.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 63.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 107.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 134.3 MB/s  0:00:00
   ━━

In [ ]:
import yaml
import torch
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM
from datasets import load_dataset, Dataset, load_from_disk

from make_prompts import generate_prompts
from load_model import load_qlora_model, load_trained_model

with open("config/sft_config.yaml", "r", encoding="utf-8") as file:
    cfg = yaml.safe_load(file)

### Inference

In [2]:
# load test data
test_data = load_from_disk("data/test_data_with_reasoning")
train_data = load_from_disk("data/train_data_with_reasoning")

In [3]:
# load model and tokenizer
adapter_path = "lora_checkpoints/sft"
MODEL_NAME = cfg["model_name"]

sft_model = load_trained_model(MODEL_NAME, adapter_path)
sft_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [4]:
# add think token to tokenizer
sft_tokenizer.add_special_tokens(
    {"additional_special_tokens": ["<|think_start|>", "<|think_end|>"]}
)
sft_model.resize_token_embeddings(len(sft_tokenizer))

Embedding(151667, 3584)

In [5]:
test_pipeline = pipeline(
    "text-generation",
    model=sft_model,
    tokenizer=sft_tokenizer,
    max_new_tokens=cfg["generation"]["max_new_tokens"],
)
sft_model.generation_config.max_length = None
# train_ds = generate_prompts(train_data[:10], sft_tokenizer, is_test=True)

# idx = 4
# # generation
# outputs = test_pipeline(
#     train_ds[idx]["text"],
#     do_sample=True,
#     temperature=cfg["generation"]["temperature"],
#     top_p=cfg["generation"]["top_p"],
#     add_special_tokens=True,
#     return_full_text=False,
# )

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [27]:
print(outputs[0]["generated_text"])

<|think_start|>
Expand around each center (both single-character and two-character centers) to find the longest palindrome whose center is at that position. For each candidate center, extend outward while the characters on both sides match until the expansion cannot continue or reaches the end of the string. Track the largest palindrome found during this process and return it. This approach exploits the palindrome property that if a substring is a palindrome then its center (or the midpoint between centers for even-length palindromes) must match itself, so checking contiguous matching characters yields the longest palindrome centered there; examining all centers guarantees discovery of the global maximum.
<|think_end|>

```python
class Solution:
    def longestPalindrome(self, s: str) -> str:
        def expand_around_center(left: int, right: int) -> str:
            # Expand around the center indices while characters match
            while left >= 0 and right < len(s) and s[left] == 

In [28]:
print("question:", train_data[idx]["query"])
print("reasoning:", train_data[idx]["reasoning"])
print("response:", train_data[idx]["response"])
# test_ds = generate_prompts(test_data, sft_tokenizer, is_test=False)
# print(test_ds[2]["text"])

question: You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

### Question:
Given a string s, return the longest palindromic substring in s.
 
Example 1:

Input: s = "babad"
Output: "bab"
Explanation: "aba" is also a valid answer.

Example 2:

Input: s = "cbbd"
Output: "bb"

 
Constraints:

1 <= s.length <= 1000
s consist of only digits and English letters.



### Format: You will use the following starter code to write the solution to the problem and enclose your code within delimiters.
```python
class Solution:
    def longestPalindrome(self, s: str) -> str:
        
```

### Answer: (use the provided format with backticks)

reasoning: Maintain a 2D table that records whether each substring is a palindrome. Use the base facts that single characters are palindromes and two-character substrings are palindromes exactly when their characters match; for lo

### Get DPO dataset

In [6]:
def get_rejected(batch):
    prompts = [
        sft_tokenizer.apply_chat_template(
            [{"role": "user", "content": q}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for q in batch["query"]
    ]
    outputs = test_pipeline(
        prompts,
        do_sample=True,
        temperature=cfg["generation"]["temperature"],
        top_p=cfg["generation"]["top_p"],
        add_special_tokens=True,
        return_full_text=False,
    )
    rejected = [out[0]["generated_text"] for out in outputs]
    return {"rejected": rejected}

In [7]:
dpo_train_data = train_data.map(
    get_rejected, batched=True, batch_size=4, load_from_cache_file=False
)

dpo_train_data.save_to_disk("data/dpo/train_data")

Map:   0%|          | 0/2641 [00:00<?, ? examples/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes

Saving the dataset (0/1 shards):   0%|          | 0/2641 [00:00<?, ? examples/s]

In [8]:
dpo_train_data = load_from_disk("data/dpo/train_data")

In [24]:
print(dpo_train_data[42]["query"])
print(dpo_train_data[42]["rejected"])

You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

### Question:
Given two non-negative integers num1 and num2 represented as strings, return the product of num1 and num2, also represented as a string.
Note: You must not use any built-in BigInteger library or convert the inputs to integer directly.
 
Example 1:
Input: num1 = "2", num2 = "3"
Output: "6"
Example 2:
Input: num1 = "123", num2 = "456"
Output: "56088"

 
Constraints:

1 <= num1.length, num2.length <= 200
num1 and num2 consist of digits only.
Both num1 and num2 do not contain any leading zero, except the number 0 itself.



### Format: You will use the following starter code to write the solution to the problem and enclose your code within delimiters.
```python
class Solution:
    def multiply(self, num1: str, num2: str) -> str:
        
```

### Answer: (use the provided format with backtick